In [ ]:
import random
import pandas as pd
from spotipy import Spotify
from spotipy.oauth2 import SpotifyClientCredentials
import os

In [ ]:
# Load the fallback dataset (adjust the path as needed)
csv_path = "datasets/moodylyrics_sample.csv"
df = pd.read_csv(csv_path)

In [ ]:
# Setup Spotify API
SPOTIFY_CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
SPOTIFY_CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

In [ ]:
if SPOTIFY_CLIENT_ID and SPOTIFY_CLIENT_SECRET:
    sp = Spotify(auth_manager=SpotifyClientCredentials(
        client_id=SPOTIFY_CLIENT_ID,
        client_secret=SPOTIFY_CLIENT_SECRET
    ))
else:
    sp = None

# Mood-based song matcher
def match_song_by_mood(mood: str) -> str:
    mood = mood.lower()

    if sp:
        try:
            search_query = {
                'happy': 'happy upbeat',
                'sad': 'sad piano',
                'energetic': 'dance pop',
                'romantic': 'romantic love',
                'calm': 'ambient chill'
            }.get(mood, 'pop')

            playlist = sp.search(q=mood, type="playlist", limit=1)
            items = playlist['playlists']['items']
            if items:
                playlist_link = items[0]['external_urls']['spotify']
                playlist_name = items[0]['name']
                return f"\Here’s a **{mood}** playlist: [{playlist_name}]({playlist_link})"
            else:
                return f"\Sorry, I couldn't find a {mood} playlist right now."
        except Exception as e:
            print("Spotify search failed. Using CSV fallback. Error:", e)

    # Fallback to CSV filtering
    if mood == 'happy':
        filtered = df[df['valence'] > 0.7]
    elif mood == 'sad':
        filtered = df[df['valence'] < 0.3]
    elif mood == 'energetic':
        filtered = df[(df['valence'] > 0.6) & (df['energy'] > 0.6)]
    else:
        filtered = df.sample(3)

    song = filtered.sample(1).iloc[0]
    return f"\Here's a recommended **{mood}** song: '{song['track_name']}' by {song['artist']}'"